# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaPrakash-Kaizu07/Flyrank-AI-intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from huggingface_hub import list_repo_files
import os

# The secret is auto-injected as an env var
hf_token = os.getenv('HF_TOKEN')

# List all files/tables in the dataset
files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

# Print table names (they're usually .parquet files)
for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [ ]:
from huggingface_hub import login
import os

hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)

import pandas as pd

# Test load
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)
print(f"✓ Loaded {len(df_march)} rows from March 2026")
print(f"Columns: {df_march.columns.tolist()}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


✓ Loaded 9841378 rows from March 2026
Columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [3]:
import os
import pandas as pd

# Auto-injected from Secret
hf_token = os.getenv('HF_TOKEN')

# Load a specific month's data (e.g., March 2026 for development)
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet",
    storage_options={"token": hf_token}
)
df_april = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet",
    storage_options={"token": hf_token}
)

# Load page metadata (dimensions)
df_content = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet",
    storage_options={"token": hf_token}
)

# Load 90-day query trend aggregation
df_query = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet",
    storage_options={"token": hf_token}
)

print(df_march.head())
print(df_april.head())
print(df_content.head())
print(df_query.head())

  report_date           client_hash_id           content_hash_id  \
0  2026-03-01  client_73cda7b4e4f265ea  content_b7e512995f79d5a6   
1  2026-03-01  client_73cda7b4e4f265ea  content_05597932fe4da067   
2  2026-03-01  client_73cda7b4e4f265ea  content_7a105f548d9c6916   
3  2026-03-01  client_73cda7b4e4f265ea  content_905aa32a0230694e   
4  2026-03-01  client_73cda7b4e4f265ea  content_a3ea9792f793ec72   

   client_has_gsc  client_has_ga4  gsc_data_available ga4_data_available  \
0            True           False                True               None   
1            True           False                True               None   
2            True           False                True               None   
3            True           False                True               None   
4            True           False                True               None   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               20           0                67  ...          N

In [4]:
# Aggregate March to (client, page) level
march_agg = df_march.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_clicks': 'sum',
    'ga4_pageviews': 'sum',
    'ga4_sessions': 'sum',
    'gsc_data_available': 'any',  # At least one day had data
}).reset_index()

# Do the same for April
april_agg = df_april.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_clicks': 'sum',
    'ga4_pageviews': 'sum',
    'ga4_sessions': 'sum',
    'gsc_data_available': 'any',
}).reset_index()

# Now find overlap at aggregated level
march_pages_agg = set(march_agg[march_agg['gsc_data_available']==True]['content_hash_id'].unique())
april_pages_agg = set(april_agg[april_agg['gsc_data_available']==True]['content_hash_id'].unique())
overlap_agg = march_pages_agg & april_pages_agg

print(f"Pages in March (agg): {len(march_pages_agg)}")
print(f"Pages in April (agg): {len(april_pages_agg)}")
print(f"Overlap (agg): {len(overlap_agg)}")

# Filter to overlap
df_march_stable = march_agg[march_agg['content_hash_id'].isin(overlap_agg)]
df_april_stable = april_agg[april_agg['content_hash_id'].isin(overlap_agg)]

print(f"\nStable (client, page) pairs: {len(df_march_stable)}")
print(f"With March impressions > 0: {(df_march_stable['gsc_impressions'] > 0).sum()}")


Pages in March (agg): 176738
Pages in April (agg): 194760
Overlap (agg): 158549

Stable (client, page) pairs: 158549
With March impressions > 0: 158549


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. Grain: One row = one (client_hash_id, content_hash_id) per month
2. Tables: fact_content_daily_performance (March + April), dim_content, fact_content_query_90d
3. Observation window: March 2026 | Label window: April 2026
4. Label: traffic_change_pct = (April impressions - March impressions) / March impressions
5. Exclusions: is_published=False, recently_updated (last 14 days), impressions=0

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Organize your columns into buckets

features = {
    'trend_pct': 'Calculated from (March last30 - prev30) / prev30 via query trends',
    'days_since_update': 'Calculated from TODAY - content_updated_date',
    'search_volume_mean': 'Mean gsc_impressions_90d across queries per page',
    'impression_volume': 'Sum of gsc_impressions in March',
    'engagement_rate': 'ga4_engaged_sessions / ga4_sessions (if available)',
}

label = {
    'traffic_change_pct': 'Calculated: (April impressions - March impressions) / March impressions',
}

context = {
    'client_hash_id': 'Which publisher owns this page',
    'content_hash_id': 'Unique page identifier',
    'report_date': 'Date of observation',
    'content_type': 'Article, guide, etc.',
}

excluded = {
    'is_published': 'Only include is_published=True; exclude False',
    'content_updated_date': 'Exclude if between 2026-03-19 and 2026-03-31 (last 14 days)',
    'gsc_impressions (zero)': 'Exclude if impressions_90d = 0 (no search opportunity)',
    'ga4_data_available (None)': 'Only use GSC; GA4 is sparse, skip for now',
    'scroll_events': 'Not used in this version',
    'ai_referrals': 'Not used in this version',
    'keyword-level data': 'Aggregate to page level; exclude raw query data',
}

print("=== FEATURES ===")
for field, reason in features.items():
    print(f"  {field}: {reason}")

print("\n=== LABEL ===")
for field, reason in label.items():
    print(f"  {field}: {reason}")

print("\n=== CONTEXT ===")
for field, reason in context.items():
    print(f"  {field}: {reason}")

print("\n=== EXCLUDED ===")
for field, reason in excluded.items():
    print(f"  {field}: {reason}")


=== FEATURES ===
  trend_pct: Calculated from (March last30 - prev30) / prev30 via query trends
  days_since_update: Calculated from TODAY - content_updated_date
  search_volume_mean: Mean gsc_impressions_90d across queries per page
  impression_volume: Sum of gsc_impressions in March
  engagement_rate: ga4_engaged_sessions / ga4_sessions (if available)

=== LABEL ===
  traffic_change_pct: Calculated: (April impressions - March impressions) / March impressions

=== CONTEXT ===
  client_hash_id: Which publisher owns this page
  content_hash_id: Unique page identifier
  report_date: Date of observation
  content_type: Article, guide, etc.

=== EXCLUDED ===
  is_published: Only include is_published=True; exclude False
  content_updated_date: Exclude if between 2026-03-19 and 2026-03-31 (last 14 days)
  gsc_impressions (zero): Exclude if impressions_90d = 0 (no search opportunity)
  ga4_data_available (None): Only use GSC; GA4 is sparse, skip for now
  scroll_events: Not used in this versi

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
print("=" * 80)
print("QUERY 1: THE GRAIN")
print("=" * 80)
print("Prove: One row = one (client_hash_id, content_hash_id) per month\n")

# Verify pages overlap between March and April
march_pages = set(df_march[df_march['gsc_data_available']==True]['content_hash_id'].unique())
april_pages = set(df_april[df_april['gsc_data_available']==True]['content_hash_id'].unique())
overlap = march_pages & april_pages

print(f"Pages in March: {len(march_pages):,}")
print(f"Pages in April: {len(april_pages):,}")
print(f"Pages in BOTH (stable): {len(overlap):,}")

QUERY 1: THE GRAIN
Prove: One row = one (client_hash_id, content_hash_id) per month

Pages in March: 176,738
Pages in April: 194,760
Pages in BOTH (stable): 158,549


In [7]:
print("\n" + "=" * 80)
print("QUERY 2: ROW COUNT & DATE SPAN")
print("=" * 80)
print("Prove: Your slice's row count and date range after aggregation\n")

# Aggregate daily data to (client, page) level
march_agg = df_march.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_clicks': 'sum',
    'gsc_data_available': 'any',
    'report_date': ['min', 'max'],
}).reset_index()
march_agg.columns = ['client_hash_id', 'content_hash_id', 'gsc_impressions',
                     'gsc_clicks', 'gsc_data_available', 'date_min', 'date_max']

april_agg = df_april.groupby(['client_hash_id', 'content_hash_id']).agg({
    'gsc_impressions': 'sum',
    'gsc_clicks': 'sum',
    'gsc_data_available': 'any',
    'report_date': ['min', 'max'],
}).reset_index()
april_agg.columns = ['client_hash_id', 'content_hash_id', 'gsc_impressions',
                     'gsc_clicks', 'gsc_data_available', 'date_min', 'date_max']

# Filter to stable pages
df_march_stable = march_agg[march_agg['content_hash_id'].isin(overlap)]
df_april_stable = april_agg[april_agg['content_hash_id'].isin(overlap)]

print(f"Stable (client, page) pairs: {len(df_march_stable):,}")
print(f"\nMarch:")
print(f"  Date range: {df_march_stable['date_min'].min()} to {df_march_stable['date_max'].max()}")
print(f"  Total rows: {len(df_march_stable):,}")
print(f"\nApril:")
print(f"  Date range: {df_april_stable['date_min'].min()} to {df_april_stable['date_max'].max()}")
print(f"  Total rows: {len(df_april_stable):,}")


QUERY 2: ROW COUNT & DATE SPAN
Prove: Your slice's row count and date range after aggregation

Stable (client, page) pairs: 158,549

March:
  Date range: 2026-03-01 to 2026-03-31
  Total rows: 158,549

April:
  Date range: 2026-04-01 to 2026-04-30
  Total rows: 158,549


In [8]:
print("\n" + "=" * 80)
print("QUERY 3: AVAILABILITY (IS TRUE CHECK)")
print("=" * 80)
print("Prove: Filter with gsc_data_available=True and show row survival\n")

available_march = df_march_stable[df_march_stable['gsc_data_available']==True]
available_april = df_april_stable[df_april_stable['gsc_data_available']==True]

print(f"March rows with gsc_data_available=True: {len(available_march):,}")
print(f"March rows with impressions > 0: {(available_march['gsc_impressions'] > 0).sum():,}")
print(f"March rows with clicks > 0: {(available_march['gsc_clicks'] > 0).sum():,}")

print(f"\nApril rows with gsc_data_available=True: {len(available_april):,}")
print(f"April rows with impressions > 0: {(available_april['gsc_impressions'] > 0).sum():,}")
print(f"April rows with clicks > 0: {(available_april['gsc_clicks'] > 0).sum():,}")


QUERY 3: AVAILABILITY (IS TRUE CHECK)
Prove: Filter with gsc_data_available=True and show row survival

March rows with gsc_data_available=True: 158,549
March rows with impressions > 0: 158,549
March rows with clicks > 0: 68,040

April rows with gsc_data_available=True: 158,549
April rows with impressions > 0: 158,549
April rows with clicks > 0: 59,221


In [9]:
print("\n" + "=" * 80)
print("FIVE FEATURES (with availability reasoning)")
print("=" * 80)

# Join March and April to calculate label
df_features = df_march_stable[['client_hash_id', 'content_hash_id',
                                'gsc_impressions', 'gsc_clicks']].rename(
    columns={'gsc_impressions': 'march_impressions', 'gsc_clicks': 'march_clicks'}
).merge(
    df_april_stable[['client_hash_id', 'content_hash_id',
                     'gsc_impressions', 'gsc_clicks']].rename(
        columns={'gsc_impressions': 'april_impressions', 'gsc_clicks': 'april_clicks'}
    ),
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

# Join with content metadata
df_features = df_features.merge(df_content[['content_hash_id', 'content_updated_date']],
                                on='content_hash_id', how='left')

# Join with query trends (aggregate to page level)
query_agg = df_query.groupby('content_hash_id').agg({
    'impressions_90d': 'mean',
    'impressions_last30': 'mean',
    'impressions_prev30': 'mean',
    'clicks_90d': 'mean',
    'avg_position_90d': 'mean',
}).reset_index()
df_features = df_features.merge(query_agg, on='content_hash_id', how='left')

print("\n--- FEATURE 1: trend_pct ---")
print("Knowable at decision moment because: Query trend data is published daily")
print("                                     We observe last30 vs prev30 in the 90d window\n")

df_features['trend_pct'] = ((df_features['impressions_last30'] - df_features['impressions_prev30'])
                             / (df_features['impressions_prev30'] + 1))  # +1 to avoid div by zero

print(f"Calculated trend_pct")
print(f"  Min: {df_features['trend_pct'].min():.3f}")
print(f"  Max: {df_features['trend_pct'].max():.3f}")
print(f"  Mean: {df_features['trend_pct'].mean():.3f}")
print(f"  Null: {df_features['trend_pct'].isnull().sum():,}")

print("\n--- FEATURE 2: days_since_update ---")
print("Knowable at decision moment because: content_updated_date is metadata set at publish time\n")

from datetime import datetime
observation_date = pd.to_datetime('2026-03-31')
df_features['days_since_update'] = (observation_date - pd.to_datetime(df_features['content_updated_date'])).dt.days

print(f"Calculated days_since_update (relative to 2026-03-31)")
print(f"  Min: {df_features['days_since_update'].min()}")
print(f"  Max: {df_features['days_since_update'].max()}")
print(f"  Mean: {df_features['days_since_update'].mean():.1f}")
print(f"  Null: {df_features['days_since_update'].isnull().sum():,}")

print("\n--- FEATURE 3: search_volume_mean ---")
print("Knowable at decision moment because: impressions_90d is aggregated from Google Search Console\n")

df_features['search_volume_mean'] = df_features['impressions_90d']

print(f"Using impressions_90d (mean across queries per page)")
print(f"  Min: {df_features['search_volume_mean'].min():.1f}")
print(f"  Max: {df_features['search_volume_mean'].max():.1f}")
print(f"  Mean: {df_features['search_volume_mean'].mean():.1f}")
print(f"  Null: {df_features['search_volume_mean'].isnull().sum():,}")

print("\n--- FEATURE 4: ctr_trend ---")
print("Knowable at decision moment because: Click trends are published in GSC daily\n")

df_features['ctr_trend'] = ((df_features['clicks_90d'] + 1) / (df_features['impressions_90d'] + 1))

print(f"Calculated CTR trend (clicks_90d / impressions_90d)")
print(f"  Min: {df_features['ctr_trend'].min():.4f}")
print(f"  Max: {df_features['ctr_trend'].max():.4f}")
print(f"  Mean: {df_features['ctr_trend'].mean():.4f}")
print(f"  Null: {df_features['ctr_trend'].isnull().sum():,}")

print("\n--- FEATURE 5: ranking_health ---")
print("Knowable at decision moment because: avg_position is published daily in GSC\n")

df_features['ranking_health'] = 100 - df_features['avg_position_90d']  # Lower position is better

print(f"Calculated ranking_health (100 - avg_position)")
print(f"  Min: {df_features['ranking_health'].min():.1f}")
print(f"  Max: {df_features['ranking_health'].max():.1f}")
print(f"  Mean: {df_features['ranking_health'].mean():.1f}")
print(f"  Null: {df_features['ranking_health'].isnull().sum():,}")


FIVE FEATURES (with availability reasoning)

--- FEATURE 1: trend_pct ---
Knowable at decision moment because: Query trend data is published daily
                                     We observe last30 vs prev30 in the 90d window

Calculated trend_pct
  Min: -1.000
  Max: 2861.000
  Mean: 0.397
  Null: 56,497

--- FEATURE 2: days_since_update ---
Knowable at decision moment because: content_updated_date is metadata set at publish time

Calculated days_since_update (relative to 2026-03-31)
  Min: -97
  Max: 303
  Mean: -48.5
  Null: 0

--- FEATURE 3: search_volume_mean ---
Knowable at decision moment because: impressions_90d is aggregated from Google Search Console

Using impressions_90d (mean across queries per page)
  Min: 10.0
  Max: 34887.0
  Mean: 52.9
  Null: 56,497

--- FEATURE 4: ctr_trend ---
Knowable at decision moment because: Click trends are published in GSC daily

Calculated CTR trend (clicks_90d / impressions_90d)
  Min: 0.0000
  Max: 0.3125
  Mean: 0.0387
  Null: 56,497

In [10]:
print("\n" + "=" * 80)
print("CALCULATE LABEL (for validation only)")
print("=" * 80)

df_features['traffic_change_pct'] = ((df_features['april_impressions'] - df_features['march_impressions'])
                                     / (df_features['march_impressions'] + 1))

print(f"\nLabel: traffic_change_pct (April impressions vs March)")
print(f"  Min: {df_features['traffic_change_pct'].min():.3f}")
print(f"  Max: {df_features['traffic_change_pct'].max():.3f}")
print(f"  Mean: {df_features['traffic_change_pct'].mean():.3f}")
print(f"  Std: {df_features['traffic_change_pct'].std():.3f}")


CALCULATE LABEL (for validation only)

Label: traffic_change_pct (April impressions vs March)
  Min: -1.000
  Max: 4536.000
  Mean: 1.447
  Std: 25.640


In [11]:
print("\n" + "=" * 80)
print("LEAKAGE TRAP: ADD ONE LABEL-DERIVED FEATURE")
print("=" * 80)

print("\n[1] Adding leakage column: april_impressions (the outcome)\n")

df_features['april_impressions_leaked'] = df_features['april_impressions']

# Quick score with leakage
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X_with_leak = df_features[['trend_pct', 'days_since_update', 'search_volume_mean',
                            'ctr_trend', 'ranking_health', 'april_impressions_leaked']].fillna(0)
y = df_features['traffic_change_pct'].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_with_leak)
model_leak = LinearRegression()
model_leak.fit(X_scaled, y)

r2_with_leak = model_leak.score(X_scaled, y)
print(f"R² score WITH leakage (april_impressions): {r2_with_leak:.4f}")
print("^ This jumped because we're using the outcome to predict itself!\n")

print("[2] Removing leakage column\n")

# Remove leakage and recalculate
X_honest = df_features[['trend_pct', 'days_since_update', 'search_volume_mean',
                         'ctr_trend', 'ranking_health']].fillna(0)

X_scaled_honest = scaler.fit_transform(X_honest)
model_honest = LinearRegression()
model_honest.fit(X_scaled_honest, y)

r2_honest = model_honest.score(X_scaled_honest, y)
print(f"R² score WITHOUT leakage: {r2_honest:.4f}")
print("^ This is the honest score. Features explain this much of the outcome.\n")

print(f"Leakage penalty: {r2_with_leak - r2_honest:.4f}")
print("^ That's how much the label-derived feature artificially boosted the score.")

# Keep the honest model and features
df_features = df_features[['client_hash_id', 'content_hash_id',
                            'trend_pct', 'days_since_update', 'search_volume_mean',
                            'ctr_trend', 'ranking_health', 'traffic_change_pct']]

print("\n" + "=" * 80)
print(f"FINAL FEATURE FRAME: {len(df_features):,} rows × {len(df_features.columns)} columns")
print("=" * 80)
print(df_features.head(10))


LEAKAGE TRAP: ADD ONE LABEL-DERIVED FEATURE

[1] Adding leakage column: april_impressions (the outcome)

R² score WITH leakage (april_impressions): 0.0058
^ This jumped because we're using the outcome to predict itself!

[2] Removing leakage column

R² score WITHOUT leakage: 0.0058
^ This is the honest score. Features explain this much of the outcome.

Leakage penalty: 0.0000
^ That's how much the label-derived feature artificially boosted the score.

FINAL FEATURE FRAME: 158,549 rows × 8 columns
            client_hash_id           content_hash_id  trend_pct  \
0  client_0797ff3a1fc9a6a5  content_0263d5f9b7a2ecd4        NaN   
1  client_0797ff3a1fc9a6a5  content_04c67f3541177192        NaN   
2  client_0797ff3a1fc9a6a5  content_0f30e04e709c7b5d        NaN   
3  client_0797ff3a1fc9a6a5  content_1207efddce873942        NaN   
4  client_0797ff3a1fc9a6a5  content_12890868e4cdac06        NaN   
5  client_0797ff3a1fc9a6a5  content_14df4b67b008d942        NaN   
6  client_0797ff3a1fc9a6a5  

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=" * 80)
print("SECTION 4: ONE NAMED LIMITATION OF YOUR SLICE")
print("=" * 80)

limitation = """
LIMITATION: Query trend data is sparse.

56,497 pages (35.6% of the 158,549 stable set) have NO entries in
fact_content_query_90d. These are pages with low search visibility or
keywords that don't rank in GSC. As a result:

- Features 1, 3, 4, 5 (trend_pct, search_volume_mean, ctr_trend, ranking_health)
  are NULL for 35.6% of rows
- The model can only learn from pages WITH observed search signals
- New pages or niche content (low search volume) are underrepresented
- This creates selection bias: pages we CAN score well are not representative
  of the full opportunity set

Workaround for production:
- Impute missing trend features with 0 (assume "no trend data" = "no opportunity")
- OR separate the model into two paths: high-visibility (with GSC data) and
  low-visibility (fallback to recency + publish status only)
"""

print(limitation)

SECTION 4: ONE NAMED LIMITATION OF YOUR SLICE

LIMITATION: Query trend data is sparse.

56,497 pages (35.6% of the 158,549 stable set) have NO entries in 
fact_content_query_90d. These are pages with low search visibility or 
keywords that don't rank in GSC. As a result:

- Features 1, 3, 4, 5 (trend_pct, search_volume_mean, ctr_trend, ranking_health) 
  are NULL for 35.6% of rows
- The model can only learn from pages WITH observed search signals
- New pages or niche content (low search volume) are underrepresented
- This creates selection bias: pages we CAN score well are not representative 
  of the full opportunity set

Workaround for production: 
- Impute missing trend features with 0 (assume "no trend data" = "no opportunity")
- OR separate the model into two paths: high-visibility (with GSC data) and 
  low-visibility (fallback to recency + publish status only)



## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.